In [1]:
import lfox
import lfox.lattice as lat
import lfox.evolution.hmc as lhmc
import jax
import jax.numpy as jnp
import numpy as np

# Imports below require "dev" environment
import matplotlib.pyplot as plt
import lsqfit
import gvar as gv
import tqdm

# Double precision!
jax.config.update("jax_enable_x64", True)
jax.config.update("jax_threefry_partitionable", True)


In [2]:
class ScalarAction(lhmc.Action):        

    @staticmethod
    @jax.jit
    def _S(fields, params):
        print("fields", fields)
        print("Params", params)
        phi = fields[0]
        S = phi**2
        
        for ax in range(phi.d()):
            S -= 2 * params['kappa'] * phi * phi.nn_field(ax)

        S += params['lambda'] * (phi**2 - 1)**2
        return S
    
    # Exact force function instead of autodiff, for testing purposes
    @staticmethod
    @jax.jit
    def exact_force(fields, params):
        phi = fields[0]

        J = phi.nn_field(axis=0, shift=1) + phi.nn_field(axis=0, shift=-1)
        for ax in range(1,phi.d()):
            J += phi.nn_field(axis=ax, shift=1)
            J += phi.nn_field(axis=ax, shift=-1)

        F = -2 * params['kappa'] * J
        F += 2 * phi.F
        F += 4 * params['lambda'] * (phi.F**2 - 1) * phi.F

        return F

In [3]:
def show_live(verbose=False):
    if verbose:
        print("Live details: ")
        print(jax.live_arrays())
        print("------------------------------")
    
    print("Live arrays = ", len(jax.live_arrays()))

In [4]:
lat.SquareLattice(st_dims=((4,4)))
lat.HoneycombLattice(st_dims=((4,4)))

I0000 00:00:1704605606.200679       1 tfrt_cpu_pjrt_client.cc:349] TfrtCpuClient created.


HoneycombLattice(
  st_dims=(4, 4),
  _dims=(4, 4, 2),
  _bc_coords=[i64[4,4], i64[4,4]]
)

In [5]:
d = 3
Lat4 = lat.SquareLattice(st_dims=((4,)*d))
phi4 = lat.LatticeField(lattice=Lat4, F=1)
#phi4.F = np.ones_like(phi4.F)
#phi4.set_field(np.ones_like(phi4.F))

show_live()  # Expect 1: {phi4}

S4 = ScalarAction(field_names=['phi'], params={'kappa': 0.18169, 'lambda': 1.3282})

show_live()  # Expect 1: {phi4}


Live arrays =  6
Live arrays =  6


In [6]:
print(len(phi4.bc))
print(phi4.d())
print(phi4 * phi4.nn_field(0))

3
3
st_dims (4, 4, 4)
LatticeField(
  lattice=SquareLattice(
    st_dims=(4, 4, 4),
    _dims=(4, 4, 4),
    _bc_coords=[i64[4,4,4], i64[4,4,4], i64[4,4,4]]
  ),
  F=f64[4,4,4],
  bc=(1, 1, 1),
  indices=()
)


In [7]:

HMC = lhmc.HMCRewrite(
    action=S4,
    seed=72345,
    fields={'phi': phi4},
    integrator=lhmc.LeapfrogIntegrator(eps=0.1, Nstep=10),
    observables={},
    save_freq=1,
)

HMC.evolve()

OK


In [8]:
print(phi4)
phi4.nn_field(0)

LatticeField(
  lattice=SquareLattice(
    st_dims=(4, 4, 4),
    _dims=(4, 4, 4),
    _bc_coords=[i64[4,4,4], i64[4,4,4], i64[4,4,4]]
  ),
  F=f64[4,4,4],
  bc=(1, 1, 1),
  indices=()
)


LatticeField(
  lattice=SquareLattice(
    st_dims=(4, 4, 4),
    _dims=(4, 4, 4),
    _bc_coords=[i64[4,4,4], i64[4,4,4], i64[4,4,4]]
  ),
  F=f64[4,4,4],
  bc=(1, 1, 1),
  indices=()
)

In [9]:
@jax.jit
def S2(fields, params):
        print("fields", fields)
        print("Params", params)
        phi = fields[0]
        S = phi**2
        
        for ax in range(phi.d()):
            S -= 2 * params['kappa'] * phi * phi.nn_field(ax)

        S += params['lambda'] * (phi**2 - 1)**2
        return S

S2([phi4], S4.params)

fields [LatticeField(
  lattice=SquareLattice(
    st_dims=(4, 4, 4),
    _dims=(4, 4, 4),
    _bc_coords=[i64[4,4,4], i64[4,4,4], i64[4,4,4]]
  ),
  F=f64[4,4,4],
  bc=(1, 1, 1),
  indices=()
)]
Params {'kappa': Traced<ShapedArray(float64[], weak_type=True)>with<DynamicJaxprTrace(level=1/0)>, 'lambda': Traced<ShapedArray(float64[], weak_type=True)>with<DynamicJaxprTrace(level=1/0)>}
st_dims (4, 4, 4)
st_dims (4, 4, 4)


LatticeField(
  lattice=SquareLattice(
    st_dims=(4, 4, 4),
    _dims=(4, 4, 4),
    _bc_coords=[i64[4,4,4], i64[4,4,4], i64[4,4,4]]
  ),
  F=f64[4,4,4],
  bc=(1, 1, 1),
  indices=()
)

In [10]:
print(phi4)
print(phi4**2)
print(S4._S([phi4], S4.params))
S4.S({'phi': phi4})

LatticeField(
  lattice=SquareLattice(
    st_dims=(4, 4, 4),
    _dims=(4, 4, 4),
    _bc_coords=[i64[4,4,4], i64[4,4,4], i64[4,4,4]]
  ),
  F=f64[4,4,4],
  bc=(1, 1, 1),
  indices=()
)
LatticeField(
  lattice=SquareLattice(
    st_dims=(4, 4, 4),
    _dims=(4, 4, 4),
    _bc_coords=[i64[4,4,4], i64[4,4,4], i64[4,4,4]]
  ),
  F=f64[4,4,4],
  bc=(1, 1, 1),
  indices=()
)
fields [LatticeField(
  lattice=SquareLattice(
    st_dims=(4, 4, 4),
    _dims=(4, 4, 4),
    _bc_coords=[i64[4,4,4], i64[4,4,4], i64[4,4,4]]
  ),
  F=f64[4,4,4],
  bc=(1, 1, 1),
  indices=()
)]
Params {'kappa': Traced<ShapedArray(float64[], weak_type=True)>with<DynamicJaxprTrace(level=1/0)>, 'lambda': Traced<ShapedArray(float64[], weak_type=True)>with<DynamicJaxprTrace(level=1/0)>}
LatticeField(
  lattice=SquareLattice(
    st_dims=(4, 4, 4),
    _dims=(4, 4, 4),
    _bc_coords=[i64[4,4,4], i64[4,4,4], i64[4,4,4]]
  ),
  F=f64[4,4,4],
  bc=(1, 1, 1),
  indices=()
)
fields [LatticeField(
  lattice=SquareLattice(
    s

Array(-5.76896, dtype=float64)

In [11]:

HMC = lhmc.HMCEvolver(
    action=S4,
    seed=72345,
    init_fields={'phi': phi4},
    integrator=lhmc.LeapfrogIntegrator(eps=0.1, Nstep=10),
)

show_live()  # Expect 3: {phi4, HMC.rng_key, HMC.pi_fields}

HMC.evolve(warmup=False)

show_live()  # Expect 4: above plus new entry in field_chain

HMC.evolve(warmup=True)

show_live()  # Expect 5: above plus new entry in field_chain

#HMC.evolve(warmup=True)

#show_live()


Live arrays =  14
fields [LatticeField(
  lattice=SquareLattice(
    st_dims=(4, 4, 4),
    _dims=(4, 4, 4),
    _bc_coords=[i64[4,4,4], i64[4,4,4], i64[4,4,4]]
  ),
  F=f64[1,4,4,4],
  bc=(1, 1, 1),
  indices=()
)]
Params {'kappa': Traced<ShapedArray(float64[], weak_type=True)>with<DynamicJaxprTrace(level=8/0)>, 'lambda': Traced<ShapedArray(float64[], weak_type=True)>with<DynamicJaxprTrace(level=8/0)>}
st_dims (4, 4, 4)
st_dims (4, 4, 4)
st_dims (4, 4, 4)
Live arrays =  16
fields [LatticeField(
  lattice=SquareLattice(
    st_dims=(4, 4, 4),
    _dims=(4, 4, 4),
    _bc_coords=[i64[4,4,4], i64[4,4,4], i64[4,4,4]]
  ),
  F=f64[1,1,4,4,4],
  bc=(1, 1, 1),
  indices=()
)]
Params {'kappa': Traced<ShapedArray(float64[], weak_type=True)>with<DynamicJaxprTrace(level=8/0)>, 'lambda': Traced<ShapedArray(float64[], weak_type=True)>with<DynamicJaxprTrace(level=8/0)>}
st_dims (4, 4, 4)
st_dims (4, 4, 4)
st_dims (4, 4, 4)
Live arrays =  17


In [12]:
jax.live_arrays()

[Array([[[[[ 0.72479398, -0.10557473,  0.91914526,  0.90789112],
           [ 0.96826624,  1.18010158,  0.86810626,  0.89227349],
           [ 1.05194023,  0.39843916,  1.00854106,  0.9686166 ],
           [ 0.831243  ,  0.60669059, -0.35390841,  1.10664274]],
 
          [[-0.3296643 ,  0.86438112,  0.99028753,  1.23625053],
           [ 1.02891239,  0.98272535,  0.84314608,  0.71589155],
           [ 0.88228456,  0.01526513,  0.74107833,  0.67592912],
           [ 0.71730456,  0.12346913,  0.69665472,  0.87327826]],
 
          [[ 0.85936663,  1.04643186,  0.86655029,  1.00479437],
           [ 1.0371643 ,  1.08021024,  1.15304365,  0.03725504],
           [ 0.77293523,  0.70902705, -1.32314317,  0.68725502],
           [ 0.9747754 ,  0.59173644,  1.10145937,  1.00444342]],
 
          [[ 0.90309651,  0.99640581,  0.73744537,  0.93954045],
           [ 0.3603557 ,  0.59801978,  0.82483673,  0.0657616 ],
           [ 1.41841146,  0.75259532, -0.16335616,  0.8471917 ],
           [ 1.0

In [14]:
HMC.__dict__

{'integrator': LeapfrogIntegrator(eps=0.1, Nstep=10),
 'monitor': {'delta_H': [-0.25304516818601885, -1.1507334466190087],
  'P_acc': [1.287941449499816, 3.160510125274647],
  'accept': [True, True]},
 'traj_init': 0,
 'traj_chain': [0, 1, 2],
 'action': ScalarAction(
   field_names=['phi'],
   params={'kappa': 0.18169, 'lambda': 1.3282},
   sub_actions=[]
 ),
 'seed': 72345,
 'rng_key': Array([ 847442375, 2892676708], dtype=uint32),
 'fields': {'phi': LatticeField(
    lattice=SquareLattice(
      st_dims=(4, 4, 4),
      _dims=(4, 4, 4),
      _bc_coords=[i64[4,4,4], i64[4,4,4], i64[4,4,4]]
    ),
    F=f64[1,1,4,4,4],
    bc=(1, 1, 1),
    indices=()
  )},
 'save_freq': 1,
 'observables': None,
 'field_chain': {'phi': [LatticeField(
     lattice=SquareLattice(
       st_dims=(4, 4, 4),
       _dims=(4, 4, 4),
       _bc_coords=[i64[4,4,4], i64[4,4,4], i64[4,4,4]]
     ),
     F=f64[4,4,4],
     bc=(1, 1, 1),
     indices=()
   ),
   LatticeField(
     lattice=SquareLattice(
       s